# 04 — Gold SIAF — Modelo Dimensional (Star Schema)
## Transformación Silver → Gold

Lee los Parquet de Silver SIAF y construye el modelo dimensional completo:

```
                  Dim_Tiempo
                 (SK_Tiempo)
                      │
   Dim_Financiamiento ┼── Dim_Municipalidad
   (SK_Financiamiento)│   (SK_Municipalidad)
                      │
          Fact_Ejecucion_Presupuestal
          ┌──────────────────────────┐
          │ SK_Tiempo                │
          │ SK_Geografia             │
          │ SK_Municipalidad         │
          │ SK_Clasificacion         │
          │ SK_Financiamiento        │
          │ SK_Calidad               │
          │ Monto_PIA/PIM/Recaudado  │
          │ Brecha, Tasa, Estado     │
          └──────────────────────────┘
                      │
   Dim_Clasificacion ─┘── Dim_Geografia
   (SK_Clasificacion)     (SK_Geografia)
```

**Fuente:** `data/silver/siaf/`  
**Destino:** `data/gold/dim_*/` y `data/gold/fact_ejecucion_presupuestal/`


In [ ]:
import sys, time
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from pyspark.sql import functions as F
from pyspark.sql.window import Window

from src.paths import SILVER, GOLD, CATEGORIAS_CSV, str_path, ensure_dirs
from src.spark_utils import get_spark, write_parquet
from core.audit.control_manager import ControlManager, ExecutionStatus

print("Imports OK")

In [ ]:
SILVER_PATH = str_path(SILVER["siaf"])
CATEGORIAS_PATH = str_path(CATEGORIAS_CSV)

MONTH_NAMES = {
    1: "ENERO", 2: "FEBRERO", 3: "MARZO", 4: "ABRIL",
    5: "MAYO", 6: "JUNIO", 7: "JULIO", 8: "AGOSTO",
    9: "SEPTIEMBRE", 10: "OCTUBRE", 11: "NOVIEMBRE", 12: "DICIEMBRE",
}

print(f"Silver source: {SILVER_PATH}")
print(f"Gold targets:")
for k in ["dim_tiempo", "dim_geografia", "dim_municipalidad", "dim_clasificacion_ingreso", "dim_financiamiento", "fact_ejecucion_presupuestal"]:
    print(f"  {k}: {str_path(GOLD[k])}")

ensure_dirs()

In [ ]:
spark = get_spark(app_name="MEF_Gold_SIAF", memory="6g")

## PARTE 1: Lectura Silver y Filtro de Categorías

In [ ]:
df = spark.read.parquet(SILVER_PATH)
n_silver = df.count()
print(f"Registros Silver SIAF: {n_silver:,}")

# Solo municipalidades (ya filtrado en Silver, pero por si acaso)
df = df.filter(F.col("NIVEL_GOBIERNO") == "M")
print(f"   Municipalidades: {df.count():,}")

In [ ]:
# ── Filtro y cruce con categorías por UBIGEO ─────────────────────────────────

if CATEGORIAS_CSV.exists():
    cat_df = spark.read.option("header", "true").csv(CATEGORIAS_PATH)
    # Corregir UBIGEO en cat_df (completar ceros a la izquierda a 6 dígitos)
    cat_df = cat_df.withColumn("UBIGEO_REF", F.lpad(F.col("UBIGEO").cast("string"), 6, "0"))
    
    cat_df = cat_df.select(
        F.col("UBIGEO_REF"),
        F.col("CATEGORIA")
    ).dropDuplicates(["UBIGEO_REF"])

    # Cruce por UBIGEO
    df = df.join(cat_df, df.UBIGEO == cat_df.UBIGEO_REF, "left")
    df = df.drop("UBIGEO_REF", "UBIGEO_csv")
    df = df.fillna({"CATEGORIA": "SIN_CATEGORIA"})

    n_con_cat = df.filter(F.col("CATEGORIA") != "SIN_CATEGORIA").count()
    print(f"Cruce categorías por UBIGEO: {n_con_cat:,}/{df.count():,} ({n_con_cat/df.count()*100:.1f}%)")
else:
    df = df.withColumn("CATEGORIA", F.lit("SIN_CATEGORIA"))
    print(" categorias.csv no encontrado — CATEGORIA='SIN_CATEGORIA'")

df = df.cache()
print("DataFrame Silver cacheado")


## PARTE 2: Construcción de Dimensiones

Las **Surrogate Keys (SK)** se generan con `row_number()` sobre ventanas ordenadas,
garantizando determinismo: el mismo dataset siempre produce los mismos SKs.

In [ ]:
# ── Dim_Tiempo ────────────────────────────────────────────────────────────────
month_map = F.create_map([F.lit(x) for kv in MONTH_NAMES.items() for x in kv])
dim_tiempo = (
    df.select("ANO_DOC", "MES_DOC").distinct()
    .withColumnRenamed("ANO_DOC", "Ano")
    .withColumnRenamed("MES_DOC", "Mes")
    .withColumn("Nombre_Mes",  month_map[F.col("Mes")])
    .withColumn("Trimestre",   F.ceil(F.col("Mes") / 3).cast("int"))
    .withColumn("Semestre",    F.when(F.col("Mes") <= 6, 1).otherwise(2))
    .withColumn("SK_Tiempo",   F.row_number().over(Window.orderBy("Ano", "Mes")))
).cache()
print(f"Dim_Tiempo: {dim_tiempo.count():,} filas")
dim_tiempo.show(5)

In [ ]:
# ── Dim_Geografia ─────────────────────────────────────────────────────────────
dim_geo = (
    df.select(
        F.col("DEPARTAMENTO_EJECUTORA").alias("Cod_Departamento"),
        F.col("DEPARTAMENTO_EJECUTORA_NOMBRE").alias("Nombre_Departamento"),
        F.col("PROVINCIA_EJECUTORA").alias("Cod_Provincia"),
        F.col("PROVINCIA_EJECUTORA_NOMBRE").alias("Nombre_Provincia"),
        F.col("DISTRITO_EJECUTORA").alias("Cod_Distrito"),
        F.col("DISTRITO_EJECUTORA_NOMBRE").alias("Nombre_Distrito"),
        F.col("UBIGEO")
    ).dropDuplicates(["Cod_Departamento", "Cod_Provincia", "Cod_Distrito"])
    .withColumn("SK_Geografia", F.row_number().over(Window.orderBy("Cod_Departamento", "Cod_Provincia", "Cod_Distrito")))
).cache()
print(f"Dim_Geografia: {dim_geo.count():,} filas")

In [ ]:
# ── Dim_Municipalidad ─────────────────────────────────────────────────────────
dim_municipalidad = (
    df.select(
        F.col("NIVEL_GOBIERNO").alias("Cod_Nivel_Gobierno"),
        F.col("NIVEL_GOBIERNO_NOMBRE").alias("Nivel_Gobierno"),
        F.col("SECTOR").alias("Cod_Sector"),
        F.col("SECTOR_NOMBRE").alias("Sector"),
        F.col("PLIEGO").alias("Cod_Pliego"),
        F.col("PLIEGO_NOMBRE").alias("Pliego"),
        F.col("EJECUTORA").alias("Cod_Ejecutora"),
        F.col("EJECUTORA_NOMBRE").alias("Ejecutora"),
        F.col("SEC_EJEC").alias("Sec_Ejecutora"),
        F.col("CATEGORIA").alias("Categoria_Municipal"),
    ).dropDuplicates(["Sec_Ejecutora"])
    .withColumn("SK_Municipalidad", F.row_number().over(Window.orderBy("Sec_Ejecutora")))
).cache()
print(f"Dim_Municipalidad: {dim_municipalidad.count():,} filas")

In [ ]:
# ── Dim_Clasificacion_Ingreso ─────────────────────────────────────────────────
dim_clas = (
    df.select(
        F.col("GENERICA").alias("Cod_Generica"),
        F.col("GENERICA_NOMBRE").alias("Generica"),
        F.col("SUBGENERICA").alias("Cod_Subgenerica"),
        F.col("SUBGENERICA_NOMBRE").alias("Subgenerica"),
        F.col("SUBGENERICA_DET").alias("Cod_Subgenerica_Det"),
        F.col("SUBGENERICA_DET_NOMBRE").alias("Subgenerica_Det"),
        F.col("ESPECIFICA").alias("Cod_Especifica"),
        F.col("ESPECIFICA_NOMBRE").alias("Especifica"),
        F.col("ESPECIFICA_DET").alias("Cod_Especifica_Det"),
        F.col("ESPECIFICA_DET_NOMBRE").alias("Especifica_Det"),
    ).dropDuplicates(["Cod_Generica", "Cod_Subgenerica", "Cod_Subgenerica_Det", "Cod_Especifica", "Cod_Especifica_Det"])
    .withColumn("SK_Clasificacion", F.row_number().over(Window.orderBy("Cod_Generica", "Cod_Subgenerica", "Cod_Especifica")))
).cache()
print(f"Dim_Clasificacion_Ingreso: {dim_clas.count():,} filas")

In [ ]:
# ── Dim_Financiamiento ────────────────────────────────────────────────────────
dim_fin = (
    df.select(
        F.col("FUENTE_FINANCIAMIENTO").alias("Cod_Fuente_Financiamiento"),
        F.col("FUENTE_FINANCIAMIENTO_NOMBRE").alias("Fuente_Financiamiento"),
        F.col("RUBRO").alias("Cod_Rubro"),
        F.col("RUBRO_NOMBRE").alias("Rubro"),
        F.col("TIPO_RECURSO").alias("Cod_Tipo_Recurso"),
        F.col("TIPO_RECURSO_NOMBRE").alias("Tipo_Recurso"),
    ).dropDuplicates(["Cod_Fuente_Financiamiento", "Cod_Rubro", "Cod_Tipo_Recurso"])
    .withColumn("SK_Financiamiento", F.row_number().over(Window.orderBy("Cod_Fuente_Financiamiento", "Cod_Rubro", "Cod_Tipo_Recurso")))
).cache()
print(f"Dim_Financiamiento: {dim_fin.count():,} filas")

## PARTE 3: Construcción de la Tabla de Hechos

La Fact se construye resolviendo las SKs a través de JOINs con cada dimensión.
El campo `SK_Calidad` clasifica la calidad del registro según sus DQ Flags:

| SK_Calidad | Significado |
|-----------|-------------|
| 1 | Sin flags — registro limpio |
| 2 | Advertencias menores (PIM<PIA, ratio alto) |
| 3 | Inconsistencias de jerarquía o catálogo |
| 4 | Error crítico (montos absurdos, registro incompleto) |

In [ ]:
# ── SK_Calidad desde DQ Flags ─────────────────────────────────────────────────
df = df.withColumn(
    "SK_Calidad",
    F.when(
        (F.col("DQ_ABSURD_AMOUNT") == 1) | (F.col("DQ_NEG_PIA") == 1) |
        (F.col("DQ_NEG_PIM") == 1) | (F.col("DQ_INCOMPLETE_RECORD") == 1), 4
    ).when(
        (F.col("DQ_BROKEN_HIERARCHY") == 1) | (F.col("DQ_INVALID_FUNDING") == 1) |
        (F.col("DQ_INVALID_GOV_LEVEL") == 1) | (F.col("DQ_INVALID_GEOGRAPHY") == 1) |
        (F.col("DQ_CATALOG_MISMATCH") == 1), 3
    ).when(
        (F.col("DQ_PIM_LT_PIA") == 1) | (F.col("DQ_NEG_REC") == 1) | (F.col("DQ_HIGH_RATIO") == 1), 2
    ).otherwise(1)
)
print("SK_Calidad calculada")
df.groupBy("SK_Calidad").count().orderBy("SK_Calidad").show()

In [ ]:
# ── Join para resolver SKs de todas las dimensiones ───────────────────────────
fact = (
    df
    .join(dim_tiempo,
          (df.ANO_DOC == dim_tiempo.Ano) & (df.MES_DOC == dim_tiempo.Mes), "left")
    .join(dim_geo,
          (df.DEPARTAMENTO_EJECUTORA == dim_geo.Cod_Departamento) &
          (df.PROVINCIA_EJECUTORA == dim_geo.Cod_Provincia) &
          (df.DISTRITO_EJECUTORA == dim_geo.Cod_Distrito), "left")
    .join(dim_municipalidad, df.SEC_EJEC == dim_municipalidad.Sec_Ejecutora, "left")
    .join(dim_clas,
          (df.GENERICA == dim_clas.Cod_Generica) &
          (df.SUBGENERICA == dim_clas.Cod_Subgenerica) &
          (df.SUBGENERICA_DET == dim_clas.Cod_Subgenerica_Det) &
          (df.ESPECIFICA == dim_clas.Cod_Especifica) &
          (df.ESPECIFICA_DET == dim_clas.Cod_Especifica_Det), "left")
    .join(dim_fin,
          (df.FUENTE_FINANCIAMIENTO == dim_fin.Cod_Fuente_Financiamiento) &
          (df.RUBRO == dim_fin.Cod_Rubro) &
          (df.TIPO_RECURSO == dim_fin.Cod_Tipo_Recurso), "left")
    .select(
        "SK_Tiempo", "SK_Geografia", "SK_Municipalidad", "SK_Clasificacion",
        "SK_Financiamiento",
        F.coalesce(F.col("SK_Calidad"), F.lit(1)).alias("SK_Calidad"),
        F.col("MONTO_PIA").cast("decimal(20,2)").alias("Monto_PIA"),
        F.col("MONTO_PIM").cast("decimal(20,2)").alias("Monto_PIM"),
        F.col("MONTO_RECAUDADO").cast("decimal(20,2)").alias("Monto_Recaudado"),
        F.col("BUDGET_GAP").cast("decimal(20,2)").alias("Brecha_Presupuestal"),
        F.when(
            F.col("PERFORMANCE_RATE").isNull() | F.isnan(F.col("PERFORMANCE_RATE")) |
            (F.col("PERFORMANCE_RATE") > 999999.999999), F.lit(None).cast("decimal(12,6)")
        ).otherwise(F.col("PERFORMANCE_RATE").cast("decimal(12,6)")).alias("Tasa_Avance"),
        F.col("PERFORMANCE_STATUS").alias("Estado_Avance"),
        F.col("ANO_DOC"),
    )
).cache()
n_fact = fact.count()
print(f"Fact_Ejecucion_Presupuestal: {n_fact:,} filas")

## PARTE 4: Escritura a Gold (Parquet)

In [ ]:
control = ControlManager(pipeline_name="gold_siaf")
execution = control.start(input_parameters={})

start_time = time.time()
try:
    print("\nEscribiendo Gold SIAF...")
    metrics = {}

    dims = [
        (dim_tiempo,        "dim_tiempo"),
        (dim_geo,           "dim_geografia"),
        (dim_municipalidad, "dim_municipalidad"),
        (dim_clas,          "dim_clasificacion_ingreso"),
        (dim_fin,           "dim_financiamiento"),
        (fact,              "fact_ejecucion_presupuestal"),
    ]

    for dim_df, key in dims:
        partition_by = ["ANO_DOC"] if key == "fact_ejecucion_presupuestal" else None
        n = write_parquet(dim_df, str_path(GOLD[key]), mode="overwrite", partition_by=partition_by)
        metrics[key] = n

    elapsed = time.time() - start_time
    control.end(
        status=ExecutionStatus.SUCCESS,
        output_summary={**metrics, "elapsed_sec": round(elapsed, 1)}
    )
    print(f"\nGold SIAF completado en {elapsed:.1f}s")

except Exception as e:
    control.log_error("GoldSIAFError", str(e))
    control.end(status=ExecutionStatus.FAILED, output_summary={"error": str(e)})
    raise

In [ ]:
# ── Resumen Gold ──────────────────────────────────────────────────────────────
print("\nResumen Gold SIAF:")
print(f"  {'Tabla':<35} {'Filas':>12}")
print("  " + "-" * 49)
for k, n in metrics.items():
    print(f"  {k:<33} {n:>12,}")